# Sesión 6 · Certeza y base de hechos

**IUSH · Sistemas expertos**

Primero se calcula un número. Después se mira la tabla donde ese número queda guardado.

Las prioridades son las de la sesión 5: R1 100, R2 90, R3 70, R4 60, R5 50, R6 10. R7 es nueva, prioridad 40.

| Cuenta | Fórmula |
|---|---|
| Una regla | aporte = CF de la regla × (el CF más bajo del SI) |
| Dos a favor | CF = CF1 + CF2 × (1 − CF1) |
| A favor y en contra | CF = (CF1 + CF2) / (1 − el menor de los dos valores absolutos) |

Si el CF más bajo del SI no pasa de **+0.20**, la regla no escribe.

| Palabra | Significado |
|---|---|
| **CF** | Factor de certeza. Un empujón entre −1 y +1. |
| **Aporte** | Lo que una regla escribe en esta pasada. |
| **Hecho inicial** | Dato que entra antes de correr el motor. |
| **Hecho inferido** | Dato que escribe una regla. |
| **Base de hechos** | El estado de este caso: filas con valor, CF, origen y clase. |
| **Recomendación** | La conclusión con el CF más alto. |
| **Prioridad** | El turno de la sesión 5. Está en ciclo.html. No entra en la fórmula. |

### Cómo usar este archivo

1. **Archivo → Guardar una copia en Drive.**
2. Corra las celdas de arriba hacia abajo.
3. Donde diga `ESCRIBA AQUÍ`, reemplaza el `0` por la fórmula.
4. Corra el motor después de tener el número en el papel.


## 0. El motor

Corra esta celda una vez. Hoy se mira la certeza y la tabla.


In [1]:
# =============================================================================
#  CERTEZA Y BASE DE HECHOS  ·  Sistemas expertos · Sesión 6 · IUSH
#
#  1. Una regla:   aporte = CF de la regla × (el CF más bajo del SI)
#  2. Dos a favor: CF = CF1 + CF2 × (1 − CF1)
#  3. A favor y en contra:
#                  CF = (CF1 + CF2) / (1 − min(|CF1|, |CF2|))
#
#  La base de hechos guarda el estado de ESTE caso:
#  hechos iniciales (entran antes) y hechos inferidos (los escribe una regla).
# =============================================================================

UMBRAL = 0.20

OPS = {
    "==": lambda a, b: a == b,
    "!=": lambda a, b: a != b,
    ">": lambda a, b: a > b,
    ">=": lambda a, b: a >= b,
    "<": lambda a, b: a < b,
    "<=": lambda a, b: a <= b,
}


def combinar_cf(cf1, cf2):
    """Junta dos certezas de la MISMA conclusión."""
    if cf1 >= 0 and cf2 >= 0:
        nuevo = cf1 + cf2 * (1 - cf1)
    elif cf1 < 0 and cf2 < 0:
        nuevo = cf1 + cf2 * (1 + cf1)
    else:
        denom = 1 - min(abs(cf1), abs(cf2))
        nuevo = 0.0 if denom == 0 else (cf1 + cf2) / denom
    return round(nuevo, 4)


def fmt_cf(cf):
    return f"{cf:+.2f}"


def fmt_valor(v):
    if isinstance(v, bool):
        return "sí" if v else "no"
    if isinstance(v, (int, float)) and not isinstance(v, bool) and abs(v) >= 1000:
        return f"{v:,.0f}".replace(",", ".")
    if isinstance(v, float) and v == int(v):
        return str(int(v))
    return str(v)


def texto_combinacion(antes, aporte):
    despues = combinar_cf(antes, aporte)
    if antes >= 0 and aporte >= 0:
        cuerpo = f"{antes:.2f} + {aporte:.2f} × (1 − {antes:.2f})"
    elif antes < 0 and aporte < 0:
        cuerpo = f"{antes:.2f} + ({aporte:.2f}) × (1 + ({antes:.2f}))"
    else:
        menor = min(abs(antes), abs(aporte))
        cuerpo = f"({antes:.2f} + ({aporte:.2f})) / (1 − {menor:.2f})"
    return f"{cuerpo} = {despues:.2f}"


class Hecho:
    """Una fila de la base de hechos."""

    def __init__(self, nombre, valor, cf=1.0, origen="", clase="inicial"):
        self.nombre = nombre
        self.valor = valor
        self.cf = round(float(cf), 4)
        self.origen = origen
        self.clase = clase  # "inicial" o "inferido"

    def linea(self):
        return (f"{self.clase:<10} {self.nombre:<22} {fmt_valor(self.valor):<16} "
                f"{fmt_cf(self.cf):<8} {self.origen}")


class Regla:
    def __init__(self, id, si, entonces, prioridad=10, cf=1.0, porque=""):
        self.id = id
        self.si = [tuple(c) for c in si]
        self.entonces = tuple(entonces)
        self.prioridad = prioridad
        self.cf = cf
        self.porque = porque

    @property
    def especificidad(self):
        return len(self.si)


class Constructor:
    def __init__(self, id):
        self._id = id
        self._si = []
        self._entonces = None
        self._prioridad = 10
        self._cf = 1.0
        self._porque = ""

    def si(self, hecho, op, valor):
        if op not in OPS:
            raise ValueError(f"Operador desconocido: {op}")
        self._si.append((hecho, op, valor))
        return self

    def entonces(self, hecho, valor):
        self._entonces = (hecho, valor)
        return self

    def prioridad(self, n):
        self._prioridad = int(n)
        return self

    def cf(self, n):
        self._cf = float(n)
        return self

    def porque(self, texto):
        self._porque = texto
        return self

    def construir(self):
        if not self._si or self._entonces is None:
            raise ValueError(f"{self._id}: faltan el SI o el ENTONCES")
        return Regla(self._id, self._si, self._entonces, self._prioridad, self._cf, self._porque)


class BaseHechos:
    """Estado de un caso. Mañana, con otro viernes, esta tabla se borra."""

    def __init__(self, caso=""):
        self.caso = caso
        self.hechos = []

    def anotar_inicial(self, nombre, valor, cf=1.0, origen="dato"):
        self.hechos = [h for h in self.hechos if not (h.nombre == nombre and h.clase == "inicial")]
        self.hechos.append(Hecho(nombre, valor, cf, origen, "inicial"))

    def inicial(self, nombre):
        for h in self.hechos:
            if h.nombre == nombre and h.clase == "inicial":
                return h
        return None

    def buscar_inferido(self, nombre, valor):
        for h in self.hechos:
            if h.nombre == nombre and h.valor == valor and h.clase == "inferido":
                return h
        return None

    def anotar_inferido(self, nombre, valor, aporte, regla_id):
        """Escribe o combina. Devuelve el relato del paso."""
        previo = self.buscar_inferido(nombre, valor)
        if previo is None:
            self.hechos.append(Hecho(nombre, valor, aporte, regla_id, "inferido"))
            return {
                "antes": None,
                "aporte": aporte,
                "despues": round(aporte, 4),
                "formula": f"{aporte:.2f} queda en la fila",
                "origen": regla_id,
            }
        antes = previo.cf
        nuevo = combinar_cf(antes, aporte)
        previo.cf = nuevo
        previo.origen = f"{previo.origen} y {regla_id}"
        return {
            "antes": antes,
            "aporte": aporte,
            "despues": nuevo,
            "formula": texto_combinacion(antes, aporte),
            "origen": previo.origen,
        }

    def inferidos(self, nombre="decision"):
        return [h for h in self.hechos if h.clase == "inferido" and h.nombre == nombre]

    def recomendacion(self, nombre="decision"):
        filas = self.inferidos(nombre)
        if not filas:
            return None
        mejor = max(filas, key=lambda h: h.cf)
        if mejor.cf <= UMBRAL:
            return None
        return mejor

    def tabla(self):
        lineas = [f"{'clase':<10} {'hecho':<22} {'valor':<16} {'CF':<8} origen"]
        lineas.append("-" * 72)
        for h in self.hechos:
            lineas.append(h.linea())
        return "\n".join(lineas)


class Motor:
    def __init__(self, reglas, nombre="Sistema experto"):
        self.nombre = nombre
        self.bc = list(reglas)
        self.bf = BaseHechos()
        self.disparadas = []
        self.pasos = []

    def cargar(self, hechos, caso=""):
        self.bf = BaseHechos(caso)
        self.disparadas = []
        self.pasos = []
        for nombre, info in hechos.items():
            if isinstance(info, dict):
                self.bf.anotar_inicial(
                    nombre, info["valor"], info.get("cf", 1.0), info.get("origen", "dato"))
            else:
                self.bf.anotar_inicial(nombre, info, 1.0, "dato")
        return self

    def cf_condicion(self, cond):
        nombre, op, esperado = cond
        h = self.bf.inicial(nombre)
        if h is None:
            return 0.0
        try:
            if not OPS[op](h.valor, esperado):
                return 0.0
        except TypeError:
            return 0.0
        return h.cf

    def cf_premisas(self, regla):
        if not regla.si:
            return 1.0
        return min(self.cf_condicion(c) for c in regla.si)

    def agenda(self):
        candidatas = []
        for r in self.bc:
            if r.id in self.disparadas:
                continue
            if self.cf_premisas(r) > UMBRAL:
                candidatas.append(r)
        return candidatas

    def resolver(self, candidatas):
        if not candidatas:
            return None
        ordenadas = sorted(
            enumerate(candidatas),
            key=lambda t: (-t[1].prioridad, -t[1].especificidad, t[0]),
        )
        return ordenadas[0][1]

    def disparar(self, regla, ciclo):
        cf_si = self.cf_premisas(regla)
        aporte = round(regla.cf * cf_si, 4)
        nombre, valor = regla.entonces
        relato = self.bf.anotar_inferido(nombre, valor, aporte, regla.id)
        self.disparadas.append(regla.id)
        paso = {
            "ciclo": ciclo,
            "regla": regla.id,
            "porque": regla.porque,
            "cf_si": cf_si,
            "cf_regla": regla.cf,
            "aporte": aporte,
            "aporte_texto": f"{regla.cf:.2f} × {cf_si:.2f} = {aporte:.2f}",
            "valor": valor,
            "antes": relato["antes"],
            "despues": relato["despues"],
            "formula": relato["formula"],
            "origen": relato["origen"],
        }
        self.pasos.append(paso)
        return paso

    def adelante(self, max_ciclos=20, verbose=True):
        if verbose:
            print(f"--- {self.nombre} ---")
            print(self.bf.tabla())
            print()
        for ciclo in range(1, max_ciclos + 1):
            candidatas = self.agenda()
            if not candidatas:
                if verbose:
                    print(f"Ciclo {ciclo}: no queda ninguna regla. Fin.")
                break
            regla = self.resolver(candidatas)
            paso = self.disparar(regla, ciclo)
            if verbose:
                ids = ", ".join(r.id for r in candidatas)
                print(f"Ciclo {paso['ciclo']}: pueden {ids}  →  sigue {paso['regla']}")
                print(f"         aporte  {paso['aporte_texto']}")
                if paso["antes"] is None:
                    print(f"         escribe {paso['valor']}  {fmt_cf(paso['despues'])}")
                else:
                    print(f"         combina {paso['formula']}")
                print(f"         fila: {paso['valor']}  {fmt_cf(paso['despues'])}  ({paso['origen']})")
        if verbose:
            print()
            print(self.bf.tabla())
            rec = self.bf.recomendacion()
            if rec:
                print(f"\nRecomendación: {rec.valor}  {fmt_cf(rec.cf)}  ({rec.origen})")
            else:
                print("\nSin recomendación.")
        return self.bf.recomendacion()


## 1. Los tres viernes y Lucía


In [2]:
def reglas_viernes():
    """Las prioridades son las de la sesión 5, las de ciclo.html.

    R1 100, R2 90, R3 70, R4 60, R5 50, R6 10.
    R7 es nueva esta noche: prioridad 40, entre R5 y R6, CF −0.20.
    """
    return [
        (Constructor("R1")
         .si("parcial_manana", "==", True)
         .si("estudio_listo", "==", False)
         .entonces("decision", "quedarse")
         .prioridad(100)
         .cf(0.80)
         .porque("Parcial mañana y el cuaderno está en blanco.")
         .construir()),
        (Constructor("R2")
         .si("energia", "<=", 2)
         .entonces("decision", "quedarse")
         .prioridad(90)
         .cf(0.50)
         .porque("Con la energía en 2, el plan se cae.")
         .construir()),
        (Constructor("R3")
         .si("plata", "<", 20000)
         .si("hay_plan", "==", True)
         .entonces("decision", "casa_de_amigo")
         .prioridad(70)
         .cf(0.50)
         .porque("Hay plan y la plata no alcanza para salir.")
         .construir()),
        (Constructor("R4")
         .si("hay_plan", "==", True)
         .si("amigos_confirmaron", "==", True)
         .si("llueve", "==", False)
         .si("energia", ">=", 3)
         .si("plata", ">=", 20000)
         .entonces("decision", "salir")
         .prioridad(60)
         .cf(0.90)
         .porque("Hay plan, confirmaron, no llueve, hay energía y hay plata.")
         .construir()),
        (Constructor("R5")
         .si("hay_plan", "==", True)
         .si("llueve", "==", True)
         .si("energia", ">=", 4)
         .entonces("decision", "casa_de_amigo")
         .prioridad(50)
         .cf(0.60)
         .porque("Llueve, hay energía y hay plan: se arma en una casa.")
         .construir()),
        (Constructor("R7")
         .si("energia", ">=", 4)
         .si("llueve", "==", True)
         .si("hay_plan", "==", True)
         .entonces("decision", "casa_de_amigo")
         .prioridad(40)
         .cf(-0.20)
         .porque("Con energía alta y lluvia, el plan de casa se puede cancelar.")
         .construir()),
        (Constructor("R6")
         .si("hay_plan", "==", False)
         .entonces("decision", "quedarse")
         .prioridad(10)
         .cf(0.40)
         .porque("Sin plan, se queda.")
         .construir()),
    ]
CAMILO = {'caso': 'Camilo · viernes 8:05 p.m.',
 'esperado': 'quedarse',
 'cf': 0.9,
 'hechos': {'plata': {'valor': 8000, 'cf': 1.0, 'origen': 'saldo Nequi a las 7:50'},
            'energia': {'valor': 2, 'cf': 1.0, 'origen': 'él mismo: estoy muerto'},
            'parcial_manana': {'valor': True,
                               'cf': 1.0,
                               'origen': 'calendario del curso'},
            'estudio_listo': {'valor': False,
                              'cf': 1.0,
                              'origen': 'el cuaderno está en blanco'},
            'llueve': {'valor': True, 'cf': 1.0, 'origen': 'el cielo de Medellín'},
            'hay_plan': {'valor': True, 'cf': 1.0, 'origen': 'grupo de WhatsApp'},
            'amigos_confirmaron': {'valor': True,
                                   'cf': 1.0,
                                   'origen': 'tres votos por Provenza'}}}
VALENTINA = {'caso': 'Valentina · viernes 8:10 p.m.',
 'esperado': 'salir',
 'cf': 0.54,
 'hechos': {'plata': {'valor': 45000, 'cf': 1.0, 'origen': 'retiro del mediodía'},
            'energia': {'valor': 4, 'cf': 1.0, 'origen': 'durmió siesta'},
            'parcial_manana': {'valor': False,
                               'cf': 1.0,
                               'origen': 'calendario del curso'},
            'estudio_listo': {'valor': True,
                              'cf': 1.0,
                              'origen': 'entregó el taller ayer'},
            'llueve': {'valor': False, 'cf': 1.0, 'origen': 'ya escampó'},
            'hay_plan': {'valor': True, 'cf': 1.0, 'origen': 'reserva en Laureles'},
            'amigos_confirmaron': {'valor': True,
                                   'cf': 0.6,
                                   'origen': 'en el grupo faltan dos respuestas'}}}
MATEO = {'caso': 'Mateo · viernes 8:12 p.m.',
 'esperado': 'casa_de_amigo',
 'cf': 0.75,
 'hechos': {'plata': {'valor': 12000, 'cf': 1.0, 'origen': 'lo que quedó del almuerzo'},
            'energia': {'valor': 5, 'cf': 1.0, 'origen': 'salió del gym'},
            'parcial_manana': {'valor': False,
                               'cf': 1.0,
                               'origen': 'calendario del curso'},
            'estudio_listo': {'valor': True,
                              'cf': 1.0,
                              'origen': 'no tiene pendientes'},
            'llueve': {'valor': True, 'cf': 1.0, 'origen': 'está lloviendo fuerte'},
            'hay_plan': {'valor': True, 'cf': 1.0, 'origen': 'lo invitaron a un apto'},
            'amigos_confirmaron': {'valor': True,
                                   'cf': 1.0,
                                   'origen': 'ya hay gente allá'}}}
LUCIA = {'caso': 'Lucía · viernes 8:20 p.m.',
 'esperado': 'quedarse',
 'cf': 0.8,
 'hechos': {'plata': {'valor': 30000, 'cf': 1.0, 'origen': 'retiro de la tarde'},
            'energia': {'valor': 3, 'cf': 0.5, 'origen': 'dice que está regular'},
            'parcial_manana': {'valor': True,
                               'cf': 1.0,
                               'origen': 'calendario del curso'},
            'estudio_listo': {'valor': False,
                              'cf': 1.0,
                              'origen': 'le falta el taller'},
            'llueve': {'valor': False, 'cf': 1.0, 'origen': 'noche seca'},
            'hay_plan': {'valor': True, 'cf': 1.0, 'origen': 'hay un plan en el grupo'},
            'amigos_confirmaron': {'valor': True,
                                   'cf': 1.0,
                                   'origen': 'confirmaron todos'}}}


## 2. Valentina · una regla

R4 tiene CF **+0.90**. El dato más bajo del SI es la confirmación de los amigos, **+0.60**.

Escriba el producto.


In [3]:
cf_regla = 0.90
cf_si = 0.60
cf_valentina = cf_regla * cf_si
print("Valentina", round(cf_valentina, 2))


Valentina 0.54


## 3. Camilo · dos reglas a favor

R1 dejó **+0.80** en quedarse. R2 aporta **+0.50** a la misma conclusión.

Primero la multiplicación.


In [4]:
cf1 = 0.80
cf2 = 0.50
cf_camilo = cf1 + cf2 * (1 - cf1)
print("Camilo", round(cf_camilo, 2))


Camilo 0.9


## 4. Mateo · sube y después baja

R3 deja casa de amigo en **+0.50**. R5 aporta **+0.60**.

Sobre ese resultado, R7 aporta **−0.20**.


In [5]:
paso1 = 0.50 + 0.60 * (1 - 0.50)
# paso1 tiene que dar 0.80 antes de seguir
en_contra = -0.20
menor = min(abs(paso1), abs(en_contra))
cf_mateo = (paso1 + en_contra) / (1 - menor)
print("Mateo paso 1", round(paso1, 2))
print("Mateo", round(cf_mateo, 2))


Mateo paso 1 0.8
Mateo 0.75


## 5. Correr y comparar

La tabla que imprime el motor es la base de hechos.

Arriba están sus cuentas. Abajo, las filas.


In [6]:
reglas = reglas_viernes()
for caso in (VALENTINA, CAMILO, MATEO):
    print("\n" + caso["caso"])
    m = Motor(reglas, nombre=caso["caso"])
    m.cargar(caso["hechos"], caso=caso["caso"])
    m.adelante(verbose=False)
    rec = m.bf.recomendacion()
    print("Recomendación:", rec.valor, f"{rec.cf:+.2f}")
    print(m.bf.tabla())



Valentina · viernes 8:10 p.m.
Recomendación: salir +0.54
clase      hecho                  valor            CF       origen
------------------------------------------------------------------------
inicial    plata                  45.000           +1.00    retiro del mediodía
inicial    energia                4                +1.00    durmió siesta
inicial    parcial_manana         no               +1.00    calendario del curso
inicial    estudio_listo          sí               +1.00    entregó el taller ayer
inicial    llueve                 no               +1.00    ya escampó
inicial    hay_plan               sí               +1.00    reserva en Laureles
inicial    amigos_confirmaron     sí               +0.60    en el grupo faltan dos respuestas
inferido   decision               salir            +0.54    R4

Camilo · viernes 8:05 p.m.
Recomendación: quedarse +0.90
clase      hecho                  valor            CF       origen
---------------------------------------------------

## 6. Inicial e inferido

Cuente, en la tabla de Camilo, cuántas filas son iniciales y cuántas inferidas.

La recomendación es una fila inferida. El saldo del Nequi es una fila inicial.


In [7]:
m = Motor(reglas_viernes(), nombre="Camilo")
m.cargar(CAMILO["hechos"])
m.adelante(verbose=False)
iniciales = [h for h in m.bf.hechos if h.clase == "inicial"]
inferidos = [h for h in m.bf.hechos if h.clase == "inferido"]
print("iniciales", len(iniciales))
print("inferidos", len(inferidos))
for h in inferidos:
    print(f"  {h.valor:<16} {h.cf:+.2f}   {h.origen}")


iniciales 7
inferidos 2
  quedarse         +0.90   R1 y R2
  casa_de_amigo    +0.50   R3


## 7. Cambiar un dato inicial

Suba la confirmación de Valentina de **+0.60** a **+1.00**.

Escriba primero su predicción. La regla R4 sigue en +0.90. Cambia la base de hechos.


In [8]:
prediccion = 0.90 * 1.00
hechos = {k: dict(v) for k, v in VALENTINA["hechos"].items()}
hechos["amigos_confirmaron"]["cf"] = 1.00
m = Motor(reglas_viernes(), nombre="Valentina confirma")
m.cargar(hechos)
m.adelante(verbose=False)
rec = m.bf.recomendacion()
print("Su predicción", round(prediccion, 2))
print("El motor     ", f"{rec.cf:+.2f}", rec.valor)


Su predicción 0.9
El motor      +0.90 salir


## 8. Lucía

Energía = 3, con CF **+0.50**. Mañana hay parcial y el taller está pendiente.

Hay plata, no llueve y el grupo confirmó.

Prediga las dos filas inferidas antes de correr: qué valor y qué CF.


In [9]:
# Predicción, antes de correr:
# quedarse CF = 0.80 × 1.00 = 0.80
# salir    CF = 0.90 × 0.50 = 0.45

m = Motor(reglas_viernes(), nombre="Lucía")
m.cargar(LUCIA["hechos"])
m.adelante()


--- Lucía ---
clase      hecho                  valor            CF       origen
------------------------------------------------------------------------
inicial    plata                  30.000           +1.00    retiro de la tarde
inicial    energia                3                +0.50    dice que está regular
inicial    parcial_manana         sí               +1.00    calendario del curso
inicial    estudio_listo          no               +1.00    le falta el taller
inicial    llueve                 no               +1.00    noche seca
inicial    hay_plan               sí               +1.00    hay un plan en el grupo
inicial    amigos_confirmaron     sí               +1.00    confirmaron todos

Ciclo 1: pueden R1, R4  →  sigue R1
         aporte  0.80 × 1.00 = 0.80
         escribe quedarse  +0.80
         fila: quedarse  +0.80  (R1)
Ciclo 2: pueden R4  →  sigue R4
         aporte  0.90 × 0.50 = 0.45
         escribe salir  +0.45
         fila: salir  +0.45  (R4)
Ciclo 3: no queda